# Model understanding with SHAP and LIME

Source target 0 means gallstones present; the modeled positive class is 1. The fixed seed reproduces the existing 70/15/15 splitting convention, then trains on the 85% development partition. Preprocessing, RFE, and stacking internal CV learn only from development rows. These are fresh experimental models, not the saved full-data refits or the original LOPO fold models. Choose the model before inspecting test outcomes.

Kernel SHAP explains the complete pipeline on raw features in probability units; LIME supplies a local approximation. Both use development data only. Correlated body-composition fields and synthetic perturbations can make attributions unstable or implausible. Explanations describe model behavior, not causal effects or clinical advice.

## Setup
Install `../requirements.txt` in the notebook kernel environment first. Run cells in order.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "xai_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside Experimental or its notebooks folder")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display
from xai_utils import setup, explainers, shap_values, metrics
MODEL_NAME = "stacking"  # logistic, random_forest, xgboost, lightgbm, stacking
SEED = 42
SHAP_SAMPLES = 256  # Increase for more stable approximations; stacking can be slow.
LIME_SAMPLES = 3000
model, train, test, y_test, predict_proba, review, OUT = setup(MODEL_NAME, SEED)
kernel, lime = explainers(train, predict_proba, SEED)
display(pd.DataFrame([metrics(y_test, review.probability)]))
print(f"Training: {len(train)}; untouched test: {len(test)}; outputs: {OUT}")

## Global SHAP on held-out cases
Use a reproducible sample for runtime control. The ranking describes this sample, not population-level importance.

In [ ]:
explain = test.sample(min(20, len(test)), random_state=SEED)
values = shap_values(kernel, explain, SHAP_SAMPLES, SEED)
base = float(kernel.expected_value)
residual = predict_proba(explain)[:, 1] - (base + values.sum(axis=1))
print("Maximum SHAP reconstruction residual:", np.abs(residual).max())
importance = pd.Series(np.abs(values).mean(axis=0), index=test.columns).sort_values(ascending=False)
importance.rename("mean_absolute_shap").to_csv(OUT / "global_shap_importance.csv")
pd.DataFrame(values, index=explain.index, columns=test.columns).to_csv(OUT / "understanding_shap_values.csv", index_label="source_row")
shap.summary_plot(values, explain, show=False)
plt.tight_layout()
plt.savefig(OUT / "shap_summary.png", bbox_inches="tight")
plt.show()
plt.close("all")
display(importance.head(15))

## Explain one patient with both methods
The default patient is the first sampled case, chosen independently of outcomes. Change `SOURCE_ROW` to another held-out row.

In [ ]:
SOURCE_ROW = explain.index[0]
position = explain.index.get_loc(SOURCE_ROW)
local_shap = shap.Explanation(values=values[position], base_values=base,
    data=explain.loc[SOURCE_ROW].to_numpy(), feature_names=list(test.columns))
shap.plots.waterfall(local_shap, max_display=12, show=False)
plt.savefig(OUT / f"shap_patient_{SOURCE_ROW}.png", bbox_inches="tight")
plt.show()
plt.close("all")
local_lime = lime.explain_instance(test.loc[SOURCE_ROW].to_numpy(), predict_proba,
    labels=(1,), num_features=12, num_samples=LIME_SAMPLES)
display(pd.DataFrame(local_lime.as_list(label=1), columns=["condition", "lime_weight"]))
print("LIME local surrogate R²:", local_lime.score)
print("Actual model probability:", predict_proba(test.loc[[SOURCE_ROW]])[0, 1])
print("Local surrogate prediction:", local_lime.local_pred)
local_lime.save_to_file(str(OUT / f"lime_patient_{SOURCE_ROW}.html"))
local_lime.as_pyplot_figure(label=1)
plt.tight_layout()
plt.show()
plt.close("all")

## Interpretation prompts
- Which fields dominate the sampled global ranking?
- Which contributions push this patient toward or away from class 1?
- Do SHAP and LIME agree on direction? LIME weights apply to discretized conditions, so their magnitudes are not directly comparable to SHAP.
- Is the LIME fit adequate, and do conclusions survive more samples or a different background?

References: [Kernel SHAP API](https://shap.readthedocs.io/en/latest/generated/shap.KernelExplainer.html), [LIME API](https://lime-ml.readthedocs.io/en/latest/lime.html).